# 10 — Daily Black Marble observability

Reproduce the November 2013 spatial panels and regional support diagnostics for Samar–Leyte. DNB-BRDF, Gap-Filled radiance and quality flags are inspected together before interpreting changes in nighttime lights.

## Key takeaways

Compare the four dates using their shared radiance stretch. Read Gap-Filled availability alongside direct spatial support, then use the long-term completeness plots to place the event window in context.

## Run order and exports

Run the cells in order from the repository or notebooks/ folder. Figures use outputs/figures/main/, supplementary/ or diagnostics/. Tables use the matching outputs/tables/ folders. Every static export writes HTML, SVG, PNG and JPEG from the same layout. HTML includes Plotly for offline viewing. JPEG preserves the PNG's pixel dimensions and composites transparency on white.

## 1. Setup and Input Data

### 1.1 Required inputs

This notebook uses analysis-ready VNP46A2 exports for Eastern Visayas / Samar–Leyte.

| Input                                     | Repository path                                                               | Purpose                                                    |
| ----------------------------------------- | ----------------------------------------------------------------------------- | ---------------------------------------------------------- |
| November 2013 VNP46A2 all-band GeoTIFFs   | `datasets/vnp46a2/region8_allbands_nov2013/`                                  | Daily spatial diagnostics around Typhoon Haiyan.           |
| Eastern Visayas DNB-BRDF daily statistics | `datasets/vnp46a2/region8_daily_stats/EasternVisayas_DNBBRDF_daily_stats.csv` | Long-term spatial-completeness diagnostics from 2012–2024. |
| VNP46A2 band lookup table                 | `datasets/metadata/vnp46a2_band_lookup.csv`                                   | Band-order reference for the exported GeoTIFFs.            |

The November 2013 GeoTIFFs are expected to follow the filename pattern:

`VNP46A2_Region8_YYYYMMDD.tif`

In [ ]:
from io import BytesIO

from pathlib import Path
from tempfile import NamedTemporaryFile
from datetime import datetime

import os
import numpy as np
import pandas as pd
import rasterio

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.subplots as sp
import plotly.io as pio
import json

from PIL import Image as PILImage




def export_figure(
    fig, output_path, width=None, height=None, scale=3,
    export_html=True, force_date_axis=False, tickformat="%Y", static=True,
):
    """Export one figure layout as HTML, SVG, PNG and JPEG."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure = go.Figure(json.loads(pio.to_json(fig, validate=True)))
    figure.update_layout(
        width=width if width is not None else figure.layout.width or 1200,
        height=height if height is not None else figure.layout.height or 600,
    )
    if force_date_axis:
        figure.update_xaxes(type="date", tickformat=tickformat)

    if static:
        png_bytes = figure.to_image(format="png", scale=scale)
        svg_bytes = figure.to_image(format="svg", scale=1)
        with PILImage.open(BytesIO(png_bytes)) as image:
            rgba = image.convert("RGBA")
            jpeg = PILImage.new("RGB", rgba.size, "white")
            jpeg.paste(rgba, mask=rgba.getchannel("A"))
        buffer = BytesIO()
        jpeg.save(buffer, format="JPEG", quality=95, subsampling=0)
        output_path.with_suffix(".png").write_bytes(png_bytes)
        output_path.with_suffix(".svg").write_bytes(svg_bytes)
        output_path.with_suffix(".jpeg").write_bytes(buffer.getvalue())
    if export_html:
        figure.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs=True, full_html=True,
        )
    return fig

In [ ]:
# ==============================
# Project Root and Output Folders
# ==============================

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "notebooks").is_dir() and (directory / "datasets").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the kernel from the repository or its notebooks folder. "
        "The repository must contain sibling notebooks/ and datasets/ folders."
    )

DATASET_DIR = PROJECT_ROOT / "datasets"

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"Main figures: {OUTPUT_MAIN_FIG_DIR.relative_to(PROJECT_ROOT)}")
print(f"Supplementary figures: {OUTPUT_SUPP_FIG_DIR.relative_to(PROJECT_ROOT)}")
print(f"Main tables: {OUTPUT_MAIN_TABLE_DIR.relative_to(PROJECT_ROOT)}")
print(f"Supplementary tables: {OUTPUT_SUPP_TABLE_DIR.relative_to(PROJECT_ROOT)}")

## 2. Load November 2013 VNP46A2 Files

### 2.1 Load and filter daily GeoTIFFs

This section loads the November 2013 VNP46A2 GeoTIFFs and prepares the daily raster stack for visual inspection. The target period covers the days before, during, and after Typhoon Haiyan landfall on 8 November 2013.

The loaded daily records contain the observation date, seven-band raster array, and geographic coordinate vectors. These records are used in the succeeding spatial and temporal diagnostics.

In [ ]:
# ==============================
# Configuration
# ==============================

folder = DATASET_DIR / "vnp46a2" / "region8_allbands_nov2013"

file_list = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])
date_list = [f[-12:-4] for f in file_list]  # Extract YYYYMMDD from filenames


# November 1–30, 2013
target_dates = [f"201311{str(d).zfill(2)}" for d in range(1, 31)]

filtered_files = [f for f in file_list if any(d in f for d in target_dates)]

print(f"Total TIFFs: {len(file_list)}")
print(f"November files: {len(filtered_files)}")


# ==============================
# Load Daily Records
# ==============================

daily_data = []

for f in filtered_files:
    filepath = os.path.join(folder, f)

    with rasterio.open(filepath) as src:
        bands = src.read()  # (7, H, W)

        # Extract date (YYYYMMDD)
        date = f[-12:-4]

        height, width = src.height, src.width
        left, bottom, right, top = src.bounds

        # Preserve geographic coordinate system
        lons = np.linspace(left, right, width)
        lats = np.linspace(top, bottom, height)

        daily_data.append({
            "date": date,
            "bands": bands,
            "lons": lons,
            "lats": lats
        })

print(f"Loaded {len(daily_data)} daily records")
print("Keys:", daily_data[0].keys())

## 3. Single-Day Band Summary

### 3.1 Band-level diagnostic table

This section computes descriptive statistics for all seven VNP46A2 bands from a selected daily GeoTIFF.

Expected output columns:

| Column         | Description                            |
| -------------- | -------------------------------------- |
| `Band`         | VNP46A2 band name.                     |
| `Min`          | Minimum retained value.                |
| `Max`          | Maximum retained value.                |
| `Mean`         | Mean retained value.                   |
| `Median`       | Median retained value.                 |
| `Std`          | Standard deviation of retained values. |
| `Valid Pixels` | Number of retained non-missing pixels. |
| `Total Pixels` | Total number of pixels in the raster.  |

This table is used as an initial check of the exported VNP46A2 stack. Radiance bands are interpreted separately from quality and condition bands because they represent different parts of the observation process.

In [ ]:
# Band names in order of appearance (7 bands)
band_names = [
    'DNB_BRDF_Corrected_NTL',
    'DNB_Lunar_Irradiance',
    'Gap_Filled_DNB_BRDF_Corrected_NTL',
    'Latest_High_Quality_Retrieval',
    'Mandatory_Quality_Flag',
    'QF_Cloud_Mask',
    'Snow_Flag'
]



tiff_path = os.path.join(folder, "VNP46A2_Region8_20131110.tif")

summaries = []

with rasterio.open(tiff_path) as src:
    data = src.read()  # shape: (7, H, W)

    for i, band_array in enumerate(data):
        band_name = band_names[i]

        flat = band_array.flatten()
        clean = flat[~np.isnan(flat)]

        # Remove fill value for MQF
        if band_name == "Mandatory_Quality_Flag":
            clean = clean[clean != 255]

        stats = {
            "Band": band_name,
            "Min": np.min(clean) if clean.size else np.nan,
            "Max": np.max(clean) if clean.size else np.nan,
            "Mean": np.mean(clean) if clean.size else np.nan,
            "Median": np.median(clean) if clean.size else np.nan,
            "Std": np.std(clean) if clean.size else np.nan,
            "Valid Pixels": clean.size,
            "Total Pixels": flat.size
        }

        summaries.append(stats)

df_summary = pd.DataFrame(summaries)
df_summary

In [ ]:
# ==============================
# Export Table: Single-Day Band Summary
# ==============================

df_summary.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_vnp46a2_band_summary_20131110.csv",
    index=False,
)

## 4. Multi-Day Band Statistics

### 4.1 November 2013 band statistics

This section computes daily descriptive statistics for each VNP46A2 band across all November 2013 GeoTIFFs.

Expected output columns:

| Column         | Description                             |
| -------------- | --------------------------------------- |
| `Date`         | Observation date.                       |
| `Band`         | VNP46A2 band name.                      |
| `Min`          | Minimum retained value.                 |
| `Max`          | Maximum retained value.                 |
| `Mean`         | Mean retained value.                    |
| `Median`       | Median retained value.                  |
| `Std`          | Standard deviation of retained values.  |
| `Valid Pixels` | Number of retained pixels for the band. |
| `Total Pixels` | Total pixels in the analysis mask.      |

The resulting table supports the time-series diagnostics below by separating the directly observed DNB-BRDF band from the Gap-Filled and quality-related bands.


In [ ]:
tif_files = sorted([f for f in os.listdir(folder) if f.endswith('.tif')])

all_stats = []

for fname in tif_files:
    path = os.path.join(folder, fname)
    date = fname[-12:-4]  # Extract YYYYMMDD

    with rasterio.open(path) as src:
        bands = src.read()

        # Use QF_Cloud_Mask to define valid spatial region
        mask = bands[5]
        valid_mask = ~np.isnan(mask)

        for i, band in enumerate(bands):
            band_name = band_names[i]

            band_data = band.astype(np.float32)
            band_data[~valid_mask] = np.nan

            # Remove MQF fill value
            if band_name == "Mandatory_Quality_Flag":
                band_data[band_data == 255] = np.nan

            flat = band_data.flatten()
            clean = flat[~np.isnan(flat)]

            stats = {
                "Date": date,
                "Band": band_name,
                "Min": np.min(clean) if clean.size else np.nan,
                "Max": np.max(clean) if clean.size else np.nan,
                "Mean": np.mean(clean) if clean.size else np.nan,
                "Median": np.median(clean) if clean.size else np.nan,
                "Std": np.std(clean) if clean.size else np.nan,
                "Valid Pixels": clean.size,
                "Total Pixels": np.count_nonzero(valid_mask)
            }

            all_stats.append(stats)

df_stats = pd.DataFrame(all_stats)
df_stats["Date"] = pd.to_datetime(df_stats["Date"], format="%Y%m%d")

df_stats.head()

In [ ]:
# ==============================
# Export Table: November 2013 Band Statistics
# ==============================

df_stats.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_vnp46a2_band_statistics_nov2013.csv",
    index=False,
)


## 5. Observed vs Gap-Filled NTL with Spatial Completeness

### 5.1 Regional mean radiance and spatial completeness

This figure compares mean DNB-BRDF radiance, mean Gap-Filled radiance, and daily spatial completeness for November 2013. Typhoon Haiyan landfall is marked as the reference event.

Figure elements:

| Element              | Description                                                                 |
| -------------------- | --------------------------------------------------------------------------- |
| Blue line            | Mean DNB-BRDF radiance from direct observations.                            |
| Red line             | Mean Gap-Filled radiance.                                                   |
| Green bars           | Spatial completeness, expressed as the percentage of valid observed pixels. |
| Dashed vertical line | Typhoon Haiyan landfall.                                                    |

In [ ]:
# Filter Core Bands

dnb = df_stats[df_stats['Band'] == 'DNB_BRDF_Corrected_NTL'].copy()
gap = df_stats[df_stats['Band'] == 'Gap_Filled_DNB_BRDF_Corrected_NTL'].copy()

# Ensure chronological order
dnb = dnb.sort_values("Date")
gap = gap.sort_values("Date")

# Compute valid pixel ratio (%)
dnb['Valid_Ratio'] = (dnb['Valid Pixels'] / dnb['Total Pixels']) * 100
gap['Valid_Ratio'] = (gap['Valid Pixels'] / gap['Total Pixels']) * 100

# Prevent log-scale crash (radiance must be >0)
dnb.loc[dnb['Mean'] <= 0, 'Mean'] = np.nan
gap.loc[gap['Mean'] <= 0, 'Mean'] = np.nan

# ==============================
# Prepare Plot Dates for Export
# ==============================

dnb["Date_plot"] = pd.to_datetime(dnb["Date"]).dt.to_pydatetime()
gap["Date_plot"] = pd.to_datetime(gap["Date"]).dt.to_pydatetime()

haiyan_date = pd.Timestamp("2013-11-08").to_pydatetime()

In [ ]:
# --- Plot Setup ---
fig = go.Figure()

# --- Background Bars: Observed NTL Coverage ---
fig.add_trace(go.Bar(
    x=dnb['Date_plot'],
    y=dnb['Valid_Ratio'],
    name='Observed Pixel Coverage (%)',
    yaxis='y2',
    marker_color='rgba(0,128,0,0.15)',
    width=86400000,  # ~1 day in ms
    showlegend=False
))

# --- Observed (DNB-BRDF) Mean ---
fig.add_trace(go.Scatter(
    x=dnb['Date_plot'],
    y=dnb['Mean'],
    mode='lines+markers',
    name='Observed NTL (DNB-BRDF)',
    line=dict(color='blue', width=3),
    marker=dict(size=8, color='blue', line=dict(width=0))
))

# --- Gap-Filled Mean ---
fig.add_trace(go.Scatter(
    x=gap['Date'],
    y=gap['Mean'],
    mode='lines+markers',
    name='Gap-Filled NTL',
    line=dict(color='red', width=3),
    marker=dict(size=8, color='red', line=dict(width=0))
))

# --- Storm Event Annotation ---
storm_events = [
    {"date": "20131108", "name": "Haiyan Landfall"}
]

for event in storm_events:
    fig.add_vline(
        x=haiyan_date,
        line_width=2,
        line_dash="dash",
        line_color="black"
    )
    fig.add_annotation(
        x=haiyan_date,
        y=1.02,
        yref='paper',
        text=event["name"],
        showarrow=False,
        font=dict(size=18, color="black"),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        borderpad=4,
        xanchor="left"
    )


fig.update_layout(
    title=dict(
        # text='Daily Nighttime Radiance: Observed vs Gap-Filled with Pixel Coverage',
        font=dict(size=22)
    ),
    xaxis=dict(
        title_font=dict(size=18),
        tickfont=dict(size=14),
        showgrid=True,
        gridwidth=0.5
    ),
    yaxis=dict(
        title='Radiance (nW·cm⁻²·sr⁻¹)',
        type='log',
        title_font=dict(size=20),
        tickfont=dict(size=16),
        showgrid=True,
        gridwidth=0.5
    ),
    yaxis2=dict(
        title='Spatial Completeness (%)',
        overlaying='y',
        side='right',
        range=[0, 100],
        title_font=dict(size=20, color='green'),
        tickfont=dict(size=20, color='green'),
        showgrid=False
    ),
    template='plotly_white',
    height=600,
    width=1200,
    legend=dict(
        x=0.6,
        y=1.08,
        font=dict(size=20)
    ),
    margin=dict(l=80, r=80, t=100, b=80),
    paper_bgcolor='rgba(0,0,0,0)',

)

fig_s1 = fig
fig_s1.update_xaxes(type="date")

fig_s1.show()

In [ ]:
# ==============================
# Export Figure S1
# ==============================

export_figure(
    fig_s1,
    OUTPUT_SUPP_FIG_DIR / "fig_s1_regional_dnb_gapfilled_completeness_2013",
    width=1200,
    height=600,
    scale=3,
)

### 5.2 Plot discussion

The DNB-BRDF series shows the directly observed radiance signal and therefore responds more sharply to day-to-day changes in both surface conditions and observation availability. Around the Haiyan period, the observed series shows a clear disruption signal, but the spatial-completeness bars also show that valid-pixel support fluctuates strongly.

The Gap-Filled series appears smoother because missing or weakly observed periods are temporally propagated. This makes the time series visually more continuous, but it also reduces the distinction between direct observation and interpolation. For this reason, the Gap-Filled layer is useful as a comparator but is not treated as equivalent to directly observed DNB-BRDF radiance.

The key point is that daily product availability does not imply daily surface observability. Radiance must be interpreted together with spatial completeness, especially in cloud-impacted tropical regions.

## 6. Latest High-Quality Retrieval Age

### 6.1 Retrieval-age matrix

This section summarises the `Latest_High_Quality_Retrieval` band across November 2013. For each date, pixels are grouped by the number of days since their latest high-quality retrieval.

Expected matrix structure:

| Dimension | Description                                |
| --------- | ------------------------------------------ |
| Columns   | Observation dates.                         |
| Rows      | Days since latest high-quality retrieval.  |
| Values    | Pixel share within each retrieval-age bin. |

### 6.2 Retrieval-age heatmap

The heatmap shows how recently pixels were last observed under high-quality conditions. Values near the bottom of the plot indicate recent high-quality observations, while diagonal upward patterns indicate consecutive days without fresh retrievals.

In [ ]:
# ==============================
# Extract Retrieval-Day Histograms
# ==============================

histograms = {}

for date in date_list:
    file_path = os.path.join(folder, f"VNP46A2_Region8_{date}.tif")

    with rasterio.open(file_path) as src:
        highq_band = src.read(4).astype(float)   # Latest_High_Quality_Retrieval
        cloud_mask = src.read(6).astype(float)   # QF_Cloud_Mask

        valid_mask = ~np.isnan(cloud_mask)
        highq_band[~valid_mask] = np.nan

        clean = highq_band[~np.isnan(highq_band)]

        # Clamp retrieval days between 1–30
        days = np.clip(clean.astype(int), 1, 30)

        hist = (
            pd.Series(days)
            .value_counts()
            .reindex(range(1, 31), fill_value=0)
        )

        histograms[date] = hist


# ==============================
# Construct Matrix (% Pixel Share)
# ==============================

hist_matrix = pd.DataFrame(histograms)
hist_matrix.index = hist_matrix.index.astype(int)
hist_matrix = hist_matrix.sort_index(axis=1)

# Use dynamic pixel denominator (more robust than hardcoding 89614)
total_pixels = hist_matrix.sum(axis=0).max()
z = (hist_matrix.values / total_pixels) * 100
z[z == 0] = None

x_datetimes = [datetime.strptime(d, "%Y%m%d") for d in hist_matrix.columns]

# ==============================
# Plot Heatmap (Enlarged Styling)
# ==============================

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=x_datetimes,
    y=hist_matrix.index,
    colorscale='YlGnBu',
    zmin=0,
    zmax=100,
    colorbar=dict(
        title=dict(text='Pixel Share (%)', font=dict(size=22)),
        tickfont=dict(size=18),
        thickness=25,
        len=0.8
    ),
    showscale=True
))


# ==============================
# Storm Annotation (Haiyan)
# ==============================

storm_events = [
    {"date": "20131108", "name": "Haiyan Landfall"}
]

for event in storm_events:
    event_date = pd.Timestamp("2013-11-08").to_pydatetime()

    fig.add_vline(
        x=event_date + pd.Timedelta(days=0.5),
        line_width=3,
        line_dash="dash",
        line_color="blue"
    )

    fig.add_annotation(
        x=event_date,
        y=1.02,
        yref='paper',
        text=event["name"],
        showarrow=False,
        font=dict(size=20, color="blue"),
        bgcolor="white",
        bordercolor="blue",
        borderwidth=2,
        borderpad=6,
        xanchor="left"
    )


# ==============================
# Layout (Deliberately Enlarged)
# ==============================

fig.update_layout(
    title=dict(
        text='Age of Latest High-Quality Retrieval per Pixel (%)',
        font=dict(size=28)
    ),
    xaxis=dict(
        title='Observation Date',
        title_font=dict(size=22),
        tickfont=dict(size=18),
        showgrid=True
    ),
    yaxis=dict(
        title='Days Since Last High-Quality Retrieval',
        title_font=dict(size=22),
        tickfont=dict(size=18),
        showgrid=True
    ),
    height=750,
    width=1400,
    template='plotly_white',
    margin=dict(l=100, r=120, t=120, b=100),
    paper_bgcolor='rgba(0,0,0,0)'
)

fig_sx_retrieval = fig
fig_sx_retrieval.update_xaxes(type="date")
fig_sx_retrieval.show()

In [ ]:
# ==============================
# Export Supplementary Retrieval-Age Figure
# ==============================

export_figure(
    fig_sx_retrieval,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_retrieval_age_heatmap_nov2013",
    width=1300,
    height=700,
    scale=3,
)

### 6.3 Plot discussion

The retrieval-age heatmap shows that observation gaps are temporally structured. During cloudy periods, many pixels shift toward older retrieval ages, meaning that the product may be carrying forward information from earlier high-quality observations rather than recording fresh surface conditions.

This diagnostic is important during the Haiyan period because infrastructure disruption and cloud-driven observation loss can occur at the same time. The figure therefore helps separate apparent product continuity from direct observational support.

## 7. Multi-Panel Spatial Diagnostics

### 7.1 Daily DNB-BRDF, Gap-Filled, and MQF panels

This section produces daily three-panel spatial diagnostics for selected November 2013 dates.

Panel structure:

| Panel | Layer                               | Purpose                               |
| ----- | ----------------------------------- | ------------------------------------- |
| A     | `DNB_BRDF_Corrected_NTL`            | Directly observed nighttime radiance. |
| B     | `Gap_Filled_DNB_BRDF_Corrected_NTL` | Gap-filled radiance comparator.       |
| C     | `Mandatory_Quality_Flag`            | Pixel-level quality structure.        |

Quality-flag classes:

| MQF value | Description                                 |
| --------- | ------------------------------------------- |
| 0         | High-quality persistent observation.        |
| 1         | High-quality ephemeral observation.         |
| 2         | Poor-quality or cloud-affected observation. |
| 255       | Fill or no retrieval.                       |

In [ ]:
import gc
import geopandas as gpd

regions_path = DATASET_DIR / "boundaries" / "regions" / "Regions.shp"
if not regions_path.is_file():
    raise FileNotFoundError(regions_path.relative_to(PROJECT_ROOT))



### 7.2 Plot discussion

These panels show how radiance continuity and observation quality vary spatially within the same daily scene. The Gap-Filled layer may appear spatially complete even when the DNB-BRDF and MQF panels show fragmented direct observation.

The spatial panels therefore make the reliability problem visible: a daily radiance field can look complete while the direct observational support behind it remains uneven.


## 8. Reliability-Gated Daily Diagnostics

### 8.1 Majority-pixel rule

This section applies a simple spatial-completeness gate to selected daily scenes. A day is flagged as satisfying the majority-pixel rule when at least 50% of pixels contain valid direct observations.

Panel structure:

| Panel | Layer                  | Purpose                                       |
| ----- | ---------------------- | --------------------------------------------- |
| A     | DNB-BRDF radiance      | Directly observed radiance and mean radiance. |
| B     | Mandatory Quality Flag | Pixel-level quality support.                  |

Reliability classes:

| Condition  | Description                                       |
| ---------- | ------------------------------------------------- |
| `SC ≥ 50%` | The majority of the region is directly observed.  |
| `SC < 50%` | Direct observational support is limited.          |
| `SC = 0%`  | The region is effectively unobserved in DNB-BRDF. |

In [ ]:
# ==============================
# Visualization Parameters
# ==============================

vmax = 10
SC_THRESHOLD = 50  # Majority-pixel rule

SUBTITLE_FONTSIZE = 30
ANNOT_FONTSIZE = 30
CBAR_TITLE_FONTSIZE = 25
CBAR_TICK_FONTSIZE = 25


# ==============================
# Loop Over First 5 Days
# ==============================

for i in range(5):

    data = daily_data[i]

    arr_ntl = np.flipud(data['bands'][0])
    arr_gap = np.flipud(data['bands'][2])  # used for land mask
    arr_qf  = np.flipud(data['bands'][4])

    lats = data['lats']
    lons = data['lons']

    # ------------------------------
    # Land Mask (from gap-filled)
    # ------------------------------

    mask = ~np.isnan(arr_gap)
    total_land_px = mask.sum()

    valid_ntl = np.count_nonzero(~np.isnan(arr_ntl) & mask)
    pct_ntl = 100 * valid_ntl / total_land_px if total_land_px else 0

    # ------------------------------
    # Mean Radiance (land-masked)
    # ------------------------------

    mean_ntl = np.nanmean(arr_ntl[mask]) if total_land_px else np.nan
    mean_gap = np.nanmean(arr_gap[mask]) if total_land_px else np.nan

    # ------------------------------
    # MQF Processing
    # ------------------------------

    arr_qf_masked = np.copy(arr_qf)
    arr_qf_masked[np.isnan(arr_qf_masked)] = 255

    qf_classes = [0, 1, 2, 255]
    qf_labels  = ['HQ (Persistent)', 'HQ (Ephemeral)', 'LQ (Poor)', 'LQ (Fill)']
    qf_colors  = ["green", "blue", "orange", "darkgrey"]

    arr_qf_land = arr_qf_masked[mask]
    qf_counts = [(arr_qf_land == v).sum() for v in qf_classes]
    qf_percents = [100 * c / total_land_px if total_land_px else 0 for c in qf_counts]

    # Remap 255 → 3 for plotting
    arr_qf_plot = np.full(arr_qf.shape, np.nan)
    arr_qf_plot[mask] = arr_qf_masked[mask]
    arr_qf_plot[arr_qf_plot == 255] = 3

    # Ensure colorbars render
    arr_ntl_color = arr_ntl.copy()
    arr_qf_color  = arr_qf_plot.copy()
    if np.isnan(arr_ntl_color).all(): arr_ntl_color[0, 0] = 0
    if np.isnan(arr_qf_color).all():  arr_qf_color[0, 0]  = 0


    # ------------------------------
    # Subplots
    # ------------------------------

    titles = [
        'DNB_BRDF_Corrected_NTL',
        'Mandatory_Quality_Flag'
    ]

    fig = sp.make_subplots(
        rows=1, cols=2,
        shared_yaxes=True,
        subplot_titles=titles
    )


    # ------------------------------
    # A) DNB-BRDF
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=arr_ntl_color,
            x=lons,
            y=lats[::-1],
            zmin=0,
            zmax=vmax,
            colorscale='Cividis',
            colorbar=dict(
                title=dict(text='nW/cm²/sr', font=dict(size=CBAR_TITLE_FONTSIZE)),
                tickfont=dict(size=CBAR_TICK_FONTSIZE),
                len=0.3,
                thickness=30,
                x=0.35,
                y=0.85
            )
        ),
        row=1, col=1
    )

    # Mean + Comparison Annotation
    fig.add_annotation(
        text=(
            f"DNB-BRDF Mean: {mean_ntl:.2f}<br>"
            f"Gap-Filled Mean: {mean_gap:.2f}"
        ),
        xref="x domain", yref="y domain",
        x=0.02, y=0.02,
        showarrow=False,
        font=dict(size=ANNOT_FONTSIZE),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1
    )


    # ------------------------------
    # Spatial Completeness Annotation
    # ------------------------------

    sc_flag = "SC ≥ 50%" if pct_ntl >= SC_THRESHOLD else "SC < 50%"
    sc_color = "black" if pct_ntl >= SC_THRESHOLD else "red"

    fig.add_annotation(
        text=f"Spatial Completeness: {pct_ntl:.1f}%<br><b>{sc_flag}</b>",
        xref="x2 domain", yref="y2 domain",
        x=0.02, y=0.02,
        showarrow=False,
        font=dict(size=ANNOT_FONTSIZE, color=sc_color),
        bgcolor="white",
        bordercolor="black",
        borderwidth=1
    )


    # ------------------------------
    # B) MQF
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=arr_qf_color,
            x=lons,
            y=lats[::-1],
            zmin=0,
            zmax=3,
            colorscale=[
                [0.00, "green"],     [0.25, "green"],
                [0.25, "blue"],      [0.50, "blue"],
                [0.50, "orange"],    [0.75, "orange"],
                [0.75, "darkgrey"],  [1.00, "darkgrey"]
            ],
            showscale=True,
            colorbar=dict(
                title=dict(text='Quality Flag', font=dict(size=CBAR_TITLE_FONTSIZE)),
                tickfont=dict(size=CBAR_TICK_FONTSIZE),
                tickvals=[0, 1, 2, 3],
                ticktext=qf_labels,
                len=0.3,
                thickness=22,
                x=0.9,
                y=0.85
            )
        ),
        row=1, col=2
    )


    # ------------------------------
    # Enlarge Subplot Titles
    # ------------------------------

    for ann in fig.layout.annotations:
        if ann.text in titles:
            ann.font = dict(size=SUBTITLE_FONTSIZE)
            ann.y = 1.02


    # ------------------------------
    # Layout
    # ------------------------------

    fig.update_layout(
        width=1200,
        height=700,
        margin=dict(l=30, r=80, t=80, b=30),
        paper_bgcolor='rgba(0,0,0,0)',
        font=dict(size=18)
    )

    fig.update_xaxes(tickfont=dict(size=16))
    fig.update_yaxes(tickfont=dict(size=16))

    fig.show()

    # date_str = data["date"]
    # sc_label = "SCge50" if pct_ntl >= SC_THRESHOLD else "SClt50"

    # export_plotly_figure(
    #     fig,
    #     RELIABILITY_GATED_DIAG_DIR / f"vnp46a2_reliability_gated_{date_str}_{sc_label}",
    #     width=1200,
    #     height=700,
    #     scale=3,
    # )

    # print(f"Exported reliability-gated diagnostic for {date_str}.")

### 8.2 Plot discussion

The reliability-gated panels show that radiance values and observational support can diverge. On high-completeness days, the DNB-BRDF mean is supported by a larger fraction of direct observations. On low-completeness days, the observed mean becomes less stable or unavailable, while the Gap-Filled layer may still report a value.

The 50% threshold is used here as a visual majority-pixel diagnostic. Later notebooks evaluate spatial-completeness thresholds more systematically across settlement masks and NTL–load alignment metrics.

## 9. Long-Term Spatial Completeness, 2012–2024

### 9.1 Load daily DNB-BRDF statistics

This section loads the long-term daily DNB-BRDF statistics for Eastern Visayas / Samar–Leyte. These statistics are used to examine whether low spatial completeness is isolated to the Haiyan period or reflects a persistent feature of the observation record.

Expected input columns:

| Column                   | Description                      |
| ------------------------ | -------------------------------- |
| `date` or `system:index` | Observation date.                |
| `NTL_mean`               | Mean DNB-BRDF radiance.          |
| `NTL_median`             | Median DNB-BRDF radiance.        |
| `NTL_min`                | Minimum DNB-BRDF radiance.       |
| `NTL_max`                | Maximum DNB-BRDF radiance.       |
| `Valid_px`               | Number of valid observed pixels. |

### 9.2 Spatial-completeness preprocessing

Spatial completeness is computed as the percentage of valid observed pixels relative to the analysis-region pixel count. The resulting daily series is used to summarise completeness distributions and seasonal patterns across 2012–2024.

In [ ]:
# ==============================
# Load & Preprocess Data
# ==============================

DNB_stats = pd.read_csv(
    DATASET_DIR / "vnp46a2" / "region8_daily_stats" / "EasternVisayas_DNBBRDF_daily_stats.csv"
)


storm_events = [
    {"name": "Yolanda", "date": "2013-11-08"},
    {"name": "Ursula",  "date": "2019-12-24"},
    {"name": "Odette",  "date": "2021-12-16"},
    {"name": "Agaton",  "date": "2022-04-10"},
]

df = DNB_stats.copy()
df = df[df['Valid_px'] > 0].copy()
df['date'] = pd.to_datetime(df['date'])

# Restrict temporal range
df = df[(df['date'].dt.year >= 2012) & (df['date'].dt.year <= 2024)].copy()

ntl_95 = np.nanpercentile(df['NTL_mean'], 95)
ntl_min = np.nanpercentile(df['NTL_mean'], 0)

df_filtered = df[
    (df['NTL_mean'] >= ntl_min) &
    (df['NTL_mean'] <= ntl_95)
].copy()

df_filtered['date'] = pd.to_datetime(df_filtered['date'])
df_filtered = df_filtered.sort_values('date')

total_pixels = df_filtered['Valid_px'].max()
df_filtered['Valid_pct'] = df_filtered['Valid_px'] / total_pixels * 100
DNB_stats

In [ ]:
daily_completeness_export = df_filtered[
    [
        "date",
        "NTL_mean",
        "NTL_median",
        "NTL_min",
        "NTL_max",
        "Valid_px",
        "Valid_pct",
    ]
].copy()

daily_completeness_path = OUTPUT_SUPP_TABLE_DIR / "daily_dnb_brdf_spatial_completeness_2012_2024.csv"
daily_completeness_export.to_csv(daily_completeness_path, index=False)

print(f"Exported: {daily_completeness_path.relative_to(PROJECT_ROOT)}")

## 10. Distribution of Daily Spatial Completeness

### 10.1 Daily spatial-completeness histogram

This histogram summarises the distribution of daily spatial completeness across the 2012–2024 DNB-BRDF record.

Figure elements:

| Element      | Description                              |
| ------------ | ---------------------------------------- |
| X-axis       | Spatial-completeness bin.                |
| Y-axis       | Number of days.                          |
| Bar labels   | Percentage of retained days in each bin. |
| Colour scale | Relative share of days per bin.          |

In [ ]:
# ==============================
# 10% Bin Histogram (Clean Version)
# ==============================

# Use preprocessed df_filtered
valid_pct = df_filtered['Valid_pct'].dropna()
total_days = len(valid_pct)

# Fixed 10% bins
bins = np.arange(0, 110, 10)
counts, edges = np.histogram(valid_pct, bins=bins)
percents = counts / total_days * 100

# Bin centers for clean x-axis
bin_centers = edges[:-1] + 5  # center of each 10% bin

# ==============================
# Plot
# ==============================

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=counts,
        marker=dict(
            color=percents,
            colorscale='Viridis',
            colorbar=dict(
                title=dict(text='Days (%)', font=dict(size=22)),
                tickfont=dict(size=18),
                thickness=25
            ),
        ),
        text=[f"{p:.1f}%" for p in percents],
        textposition='outside'
    )
)

# ==============================
# Layout (Clean + Enlarged)
# ==============================

fig.update_layout(
    title=dict(
        text='Distribution of Daily Spatial Completeness',
        font=dict(size=32)
    ),
    xaxis=dict(
        title='Spatial Completeness (%)',
        title_font=dict(size=24),
        tickfont=dict(size=20),
        tickmode='linear',
        dtick=10,
        range=[0, 100]
    ),
    yaxis=dict(
        title='Number of Days',
        title_font=dict(size=24),
        tickfont=dict(size=20)
    ),
    height=750,
    width=1300,
    bargap=0.1,
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(size=20),
    margin=dict(l=80, r=120, t=120, b=80)
)

fig_sx_hist = fig
fig_sx_hist.show()

In [ ]:
export_figure(
    fig_sx_hist,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_spatial_completeness_histogram_2012_2024",
    width=1200,
    height=700,
    scale=3,
)

### 10.2 Plot discussion

The histogram shows that spatial completeness varies widely across the daily DNB-BRDF record. Low-completeness days occur frequently, while high-completeness days are present but not guaranteed.

This distribution supports the use of explicit completeness diagnostics. Without accounting for spatial completeness, days with very different levels of direct observational support would be treated as equally reliable in downstream time-series interpretation.

## 11. Weekly Spatial Completeness, 2012–2024

### 11.1 Weekly completeness heatmap

This heatmap summarises mean weekly spatial completeness across the 2012–2024 analysis period.

Figure elements:

| Element      | Description                       |
| ------------ | --------------------------------- |
| X-axis       | ISO week of year.                 |
| Top axis     | Approximate month reference.      |
| Y-axis       | Year.                             |
| Colour scale | Mean weekly spatial completeness. |

In [ ]:
# ==============================
# Prepare Data
# ==============================

total_pixels = df['Valid_px'].max()
df = df[df['Valid_px'] > 0].copy()
df['Spatial_Completeness'] = df['Valid_px'] / total_pixels * 100

df['year'] = df['date'].dt.year
df['week'] = df['date'].dt.isocalendar().week.astype(int)

df = df[df['week'] <= 52]

pivot_week = df.pivot_table(
    index='year',
    columns='week',
    values='Spatial_Completeness',
    aggfunc='mean'
)

# ==============================
# Month alignment (mid-month reference)
# ==============================

ref_year = 2020
mid_month = pd.date_range(f"{ref_year}-01-15", f"{ref_year}-12-15", freq="MS")
month_weeks = mid_month.isocalendar().week.astype(int)
month_labels = mid_month.strftime("%b")

# ==============================
# Plot
# ==============================

fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z=pivot_week.values,
        x=pivot_week.columns,
        y=pivot_week.index,
        colorscale="Viridis",
        xaxis="x",
        yaxis="y",
        colorbar=dict(
            title=dict(
                text="Spatial Completeness (%)",
                side="right"
            ),
            thickness=30,
            len=0.85,
            tickfont=dict(size=18)
        )
    )
)

fig.update_layout(
    title=dict(
        text="Samar–Leyte Spatial Completeness (2012–2024)",
        font=dict(size=34)
    ),

    # Bottom axis: Weeks
    xaxis=dict(
        title="Week of Year",
        title_font=dict(size=24),
        tickfont=dict(size=18),
        dtick=4,
        side="bottom"
    ),

    # Top axis: Months
    xaxis2=dict(
        overlaying="x",
        side="top",
        tickmode="array",
        tickvals=month_weeks,
        ticktext=month_labels,
        tickfont=dict(size=20),
        showgrid=False
    ),

    yaxis=dict(
        title="Year",
        title_font=dict(size=24),
        tickfont=dict(size=20)
    ),

    height=600,
    width=1400,
    margin=dict(l=120, r=160, t=140, b=120),
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(size=20)
)

fig.show()


### 11.2 Plot discussion

The heatmap shows that spatial completeness is seasonally structured across the long-term record. Reduced observability recurs across multiple years, indicating that missingness is not only event-driven but also embedded in the regional cloud regime.

This result supports the broader reliability framing of the study: in humid tropical environments, daily nighttime lights are conditionally usable, but their interpretation depends on explicit observability diagnostics.

## 12. Combined Spatial Completeness Summary

### 12.1 Weekly, seasonal, and annual completeness structure

This figure combines three views of spatial completeness:

| Panel        | Description                                        |
| ------------ | -------------------------------------------------- |
| Top panel    | Weekly mean spatial completeness across all years. |
| Centre panel | Weekly spatial completeness by year.               |
| Right panel  | Annual mean spatial completeness.                  |


In [ ]:
# ==============================
# Prepare Data
# ==============================

total_pixels = df['Valid_px'].max()
df = df[df['Valid_px'] > 0].copy()
df['Spatial_Completeness'] = df['Valid_px'] / total_pixels * 100

df['year'] = df['date'].dt.year
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df = df[df['week'] <= 52]

pivot_week = df.pivot_table(
    index='year',
    columns='week',
    values='Spatial_Completeness',
    aggfunc='mean'
)

weekly_mean = pivot_week.mean(axis=0)
weekly_std  = pivot_week.std(axis=0)

annual_mean = pivot_week.mean(axis=1)
annual_std  = pivot_week.std(axis=1)

# ==============================
# Subplot Layout
# ==============================

fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.3, 0.7],
    column_widths=[0.85, 0.15],
    specs=[[{"type": "xy"}, None],
           [{"type": "heatmap"}, {"type": "xy"}]],
    horizontal_spacing=0.02,
    vertical_spacing=0.03
)

# ------------------------------
# Heatmap
# ------------------------------

fig.add_trace(
    go.Heatmap(
        z=pivot_week.values,
        x=pivot_week.columns,
        y=pivot_week.index,
        zmin=0,
        zmax=100,
        colorscale="Viridis",
        colorbar=dict(
            orientation="v",
            x=0.8,
            y=.58,
            len=0.3,
            thickness=30,
            tickmode="array",
            # tickvals=[0, 100],
            ticktext=["0%", "100%"],
            title="SC (%)",
        )
    ),
    row=2, col=1
)

# ------------------------------
# Weekly Mean (Top)
# ------------------------------

fig.add_trace(
    go.Scatter(
        x=weekly_mean.index,
        y=weekly_mean.values,
        mode="lines+markers",
        name="Weekly Mean SC (%)",
        line=dict(width=3),
        marker=dict(size=10),
        error_y=dict(
            type="data",
            array=weekly_std.values,
            visible=True
        )
    ),
    row=1, col=1
)

# ------------------------------
# Annual Mean (Right)
# ------------------------------

fig.add_trace(
    go.Scatter(
        x=annual_mean.values,
        y=annual_mean.index,
        mode="lines+markers",
        name="Annual Mean SC (%)",
        line=dict(width=3),
        marker=dict(size=10),
        error_x=dict(
            type="data",
            array=annual_std.values,
            visible=True
        )
    ),
    row=2, col=2
)

# ==============================
# Layout Refinement
# ==============================

fig.update_layout(
    title=dict(
        text="Samar–Leyte Spatial Completeness (2012–2024)",
        font=dict(size=32)
    ),
    height=1000,
    width=1600,
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(size=18),
    margin=dict(l=100, r=120, t=140, b=100),
    legend=dict(
        x=.85,
        y=1,
        orientation="v",
        font=dict(size=16)
    )
)

# Align weekly plot with heatmap
fig.update_xaxes(matches="x2", row=1, col=1)  # match heatmap x-axis

# Clean axis labels
fig.update_xaxes(title=None, row=1, col=1)
fig.update_xaxes(title=None, row=2, col=2)

fig.update_xaxes(title="Week of Year", row=2, col=1)
fig.update_yaxes(title="Year", row=2, col=1)

# Remove duplicate tick labels


# Approximate ISO week → month mapping (mid-month placement)
month_ticks = {
    "Jan": 2,
    "Feb": 6,
    "Mar": 10,
    "Apr": 15,
    "May": 19,
    "Jun": 24,
    "Jul": 28,
    "Aug": 32,
    "Sep": 37,
    "Oct": 41,
    "Nov": 46,
    "Dec": 50
}

fig.update_xaxes(
    tickmode="array",
    tickvals=list(month_ticks.values()),
    ticktext=list(month_ticks.keys()),
    row=1, col=1,
)
# fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=2)

fig.update_yaxes(title="SC (%)", row=1, col=1)
fig.update_xaxes(title="SC (%)", row=2, col=2)

fig_s3 = fig
fig_s3.show()

In [ ]:
export_figure(
    fig_s3,
    OUTPUT_SUPP_FIG_DIR / "fig_s3_spatial_completeness_2012_2024",
    width=1500,
    height=900,
    scale=3,
)

In [ ]:
weekly_completeness_summary = pd.DataFrame({
    "week": weekly_mean.index,
    "weekly_mean_sc": weekly_mean.values,
    "weekly_std_sc": weekly_std.values,
})

annual_completeness_summary = pd.DataFrame({
    "year": annual_mean.index,
    "annual_mean_sc": annual_mean.values,
    "annual_std_sc": annual_std.values,
})

weekly_summary_path = OUTPUT_SUPP_TABLE_DIR / "weekly_spatial_completeness_summary_2012_2024.csv"
annual_summary_path = OUTPUT_SUPP_TABLE_DIR / "annual_spatial_completeness_summary_2012_2024.csv"

weekly_completeness_summary.to_csv(weekly_summary_path, index=False)
annual_completeness_summary.to_csv(annual_summary_path, index=False)

print(f"Exported: {weekly_summary_path.relative_to(PROJECT_ROOT)}")
print(f"Exported: {annual_summary_path.relative_to(PROJECT_ROOT)}")

### 12.2 Plot discussion

The combined figure shows that spatial completeness varies more strongly within years than between years. The seasonal pattern is repeated across much of the record, while annual averages remain comparatively stable.

This indicates that the dominant observability constraint is seasonal cloud structure rather than a simple long-term trend. Reliability qualification is therefore needed not only for individual disaster events, but for daily NTL analysis in this region more generally.

## 13. Notebook Outputs

### Main manuscript output

| Output | Path |
|---|---|
| Figure 3 — Example VNP46A2 DNB-BRDF, Gap-Filled, and MQF panels | `outputs/figures/main/fig_03_YYYYMMDD.*` |

### Supplementary outputs

| Output | Path |
|---|---|
| Figure S1 — Regional DNB-BRDF, Gap-Filled, and spatial completeness | `outputs/figures/supplementary/fig_s1_regional_dnb_gapfilled_completeness_2013.*` |
| Figure S3 — Long-term spatial completeness, 2012–2024 | `outputs/figures/supplementary/fig_s3_spatial_completeness_2012_2024.*` |
| Retrieval-age heatmap | `outputs/figures/supplementary/fig_sx_retrieval_age_heatmap_nov2013.*` |
| Additional daily VNP46A2 panels | `outputs/figures/supplementary/fig_sx_vnp46a2_daily_panels_YYYYMMDD.*` |
| Reliability-gated daily panels | `outputs/figures/supplementary/fig_sx_reliability_gated_daily_panels_2013.*` |
| Spatial completeness histogram | `outputs/figures/supplementary/fig_sx_spatial_completeness_histogram_2012_2024.*` |
| VNP46A2 single-day band summary | `outputs/tables/supplementary/table_sx_vnp46a2_band_summary_20131110.csv` |
| VNP46A2 November 2013 band statistics | `outputs/tables/supplementary/table_sx_vnp46a2_band_statistics_nov2013.csv` |

## 14. Figure 3 exports

Four date-specific panels use the shared 99th-percentile radiance display range and are exported to outputs/figures/main/.

In [ ]:
# ==========================================================
# R1.4 — Load inputs and define the Figure 3 builder
# ==========================================================

R14_TIFF_DIR = DATASET_DIR / "vnp46a2" / "region8_allbands_nov2013"
R14_REGIONS_PATH = regions_path

R14_DATES = {
    "20131101": {"panel": "a", "label": "1 Nov 2013"},
    "20131105": {"panel": "b", "label": "5 Nov 2013"},
    "20131110": {"panel": "c", "label": "10 Nov 2013"},
    "20131113": {"panel": "d", "label": "13 Nov 2013"},
}

R14_FIG_WIDTH = 820
R14_FIG_HEIGHT = 500
R14_DISPLAY_PERCENTILE = 99

# Rasterio band numbers are one-based.
R14_BAND_DNB = 1
R14_BAND_GAP = 3
R14_BAND_MQF = 5

R14_QF_LABELS = [
    "Persistent (0)",
    "Ephemeral (1)",
    "Poor/Cloud (2)",
    "Fill (255)",
]

R14_QF_COLORSCALE = [
    [0.0000, "green"], [0.2499, "green"],
    [0.2500, "royalblue"], [0.4999, "royalblue"],
    [0.5000, "orange"], [0.7499, "orange"],
    [0.7500, "lightgrey"], [1.0000, "lightgrey"],
]

for required_path in [R14_TIFF_DIR, R14_REGIONS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path.relative_to(PROJECT_ROOT))


def r14_find_tiff(date_key):
    matches = sorted(R14_TIFF_DIR.glob(f"*{date_key}*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one GeoTIFF for {date_key}; found {len(matches)}: {[path.name for path in matches]}"
        )
    return matches[0]


def r14_read_band(src, band_number):
    return src.read(band_number, masked=True).filled(np.nan).astype(np.float32)


def r14_append_boundary(geometry, x_values, y_values):
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "LineString":
        x_coords, y_coords = geometry.xy
        x_values.extend(x_coords)
        y_values.extend(y_coords)
        x_values.append(None)
        y_values.append(None)
        return

    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        r14_append_boundary(geometry.boundary, x_values, y_values)
        return

    if hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            r14_append_boundary(part, x_values, y_values)


R14_TIFF_BY_DATE = {
    date_key: r14_find_tiff(date_key)
    for date_key in R14_DATES
}

with rasterio.open(R14_TIFF_BY_DATE["20131101"]) as src:
    left, bottom, right, top = src.bounds

R14_LON_RANGE = [float(left), float(right)]
R14_LAT_RANGE = [float(bottom), float(top)]

regions = gpd.read_file(R14_REGIONS_PATH).to_crs(epsg=4326)
visible_regions = regions.cx[left:right, bottom:top]
if visible_regions.empty:
    raise ValueError("Regions.shp does not intersect the raster extent")

R14_BOUNDARY_X = []
R14_BOUNDARY_Y = []
for geometry in visible_regions.geometry:
    r14_append_boundary(geometry, R14_BOUNDARY_X, R14_BOUNDARY_Y)

del visible_regions

contrast_values = []
for path in R14_TIFF_BY_DATE.values():
    with rasterio.open(path) as src:
        ntl = r14_read_band(src, R14_BAND_DNB)
        gap = r14_read_band(src, R14_BAND_GAP)
    valid = np.isfinite(ntl) & np.isfinite(gap)
    contrast_values.append(ntl[valid])
    del ntl, gap, valid

contrast_values = np.concatenate(contrast_values)
R14_DISPLAY_MIN = 0.0
R14_DISPLAY_MAX = float(
    np.nanpercentile(contrast_values, R14_DISPLAY_PERCENTILE)
)
if not np.isfinite(R14_DISPLAY_MAX) or R14_DISPLAY_MAX <= 0:
    R14_DISPLAY_MAX = 1.0

R14_NTL_TICKS = np.linspace(R14_DISPLAY_MIN, R14_DISPLAY_MAX, 6)
del contrast_values
gc.collect()

In [ ]:
# ==========================================================
# R1.4 — Figure 3 builder and in-notebook preview
# ==========================================================

def build_r14_figure(date_key):

    path = R14_TIFF_BY_DATE[date_key]

    with rasterio.open(path) as src:

        ntl = np.flipud(
            r14_read_band(src, R14_BAND_DNB)
        )

        gap = np.flipud(
            r14_read_band(src, R14_BAND_GAP)
        )

        mqf = np.flipud(
            r14_read_band(src, R14_BAND_MQF)
        )

        left, bottom, right, top = src.bounds

        lons = np.linspace(
            left,
            right,
            src.width,
            dtype=np.float32,
        )

        lats = np.linspace(
            bottom,
            top,
            src.height,
            dtype=np.float32,
        )

    # ------------------------------
    # Statistics
    # ------------------------------

    land_mask = np.isfinite(gap)
    valid_ntl_mask = land_mask & np.isfinite(ntl)

    total_land_px = int(land_mask.sum())
    valid_ntl_px = int(valid_ntl_mask.sum())

    spatial_completeness = (
        100 * valid_ntl_px / total_land_px
        if total_land_px
        else np.nan
    )

    mean_ntl = float(
        np.nanmean(ntl[valid_ntl_mask])
    )

    median_ntl = float(
        np.nanmedian(ntl[valid_ntl_mask])
    )

    mean_gap = float(
        np.nanmean(gap[land_mask])
    )

    median_gap = float(
        np.nanmedian(gap[land_mask])
    )

    statistics = {
        "date": pd.to_datetime(
            date_key,
            format="%Y%m%d",
        ),
        "valid_pixels": valid_ntl_px,
        "total_land_pixels": total_land_px,
        "spatial_completeness_pct": spatial_completeness,
        "dnb_brdf_mean": mean_ntl,
        "dnb_brdf_median": median_ntl,
        "gap_filled_mean": mean_gap,
        "gap_filled_median": median_gap,
    }

    # ------------------------------
    # MQF plotting array
    # ------------------------------

    mqf_plot = np.full(
        mqf.shape,
        np.nan,
        dtype=np.float32,
    )

    mqf_land = mqf[land_mask].copy()

    mqf_land[
        ~np.isfinite(mqf_land)
    ] = 255

    mqf_land[
        mqf_land == 255
    ] = 3

    mqf_plot[
        land_mask
    ] = mqf_land

    del mqf_land

    # ------------------------------
    # Two-panel figure
    # ------------------------------

    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        horizontal_spacing=0.08,
        subplot_titles=[
            "DNB_BRDF_Corrected_NTL",
            "Mandatory_Quality_Flag",
        ],
    )

    # Retrieve subplot domains so each colorbar remains
    # inside its respective panel.
    ntl_domain = fig.layout.xaxis.domain
    mqf_domain = fig.layout.xaxis2.domain

    ntl_colorbar_x = ntl_domain[1] + 0.05
    mqf_colorbar_x = mqf_domain[1] + 0.1

    # ------------------------------
    # DNB-BRDF panel
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=ntl,
            x=lons,
            y=lats,
            zmin=R14_DISPLAY_MIN,
            zmax=R14_DISPLAY_MAX,
            colorscale="Cividis",
            hoverinfo="skip",
            colorbar=dict(
                title=dict(
                    text="nW/cm²/sr",
                    side="top",
                    font=dict(size=12),
                ),
                tickvals=R14_NTL_TICKS,
                ticktext=[
                    f"{value:.1f}"
                    for value in R14_NTL_TICKS
                ],
                tickfont=dict(size=10),
                len=0.28,
                thickness=10,
                x=ntl_colorbar_x,
                xanchor="right",
                y=0.9,
                yanchor="middle",
                outlinewidth=0,
            ),
        ),
        row=1,
        col=1,
    )

    # ------------------------------
    # Mandatory Quality Flag panel
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=mqf_plot,
            x=lons,
            y=lats,
            zmin=-0.5,
            zmax=3.5,
            colorscale=R14_QF_COLORSCALE,
            hoverinfo="skip",
            colorbar=dict(
                title=dict(
                    text="Quality Flag",
                    side="top",
                    font=dict(size=12),
                ),
                tickvals=[0, 1, 2, 3],
                ticktext=R14_QF_LABELS,
                tickfont=dict(size=10),
                len=0.28,
                thickness=10,
                x=mqf_colorbar_x,
                xanchor="right",
                y=0.9,
                yanchor="middle",
                outlinewidth=0,
            ),
        ),
        row=1,
        col=2,
    )

    # ------------------------------
    # Regional boundary overlay
    # ------------------------------

    for column in [1, 2]:

        fig.add_trace(
            go.Scatter(
                x=R14_BOUNDARY_X,
                y=R14_BOUNDARY_Y,
                mode="lines",
                line=dict(
                    color="rgba(45,45,45,0.85)",
                    width=0.8,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

    # ------------------------------
    # Center subplot titles
    # ------------------------------

    for annotation in fig.layout.annotations:

        if annotation.text in {
            "DNB_BRDF_Corrected_NTL",
            "Mandatory_Quality_Flag",
        }:
            annotation.font = dict(
                family="Arial",
                size=11,
                color="black",
            )

            annotation.xanchor = "center"
            annotation.y = 1.025

    # ------------------------------
    # Statistics inside white boxes
    # ------------------------------

    fig.add_annotation(
        text=(
            f"DNB-BRDF Mean: {mean_ntl:.2f}; "
            f"Median: {median_ntl:.2f}<br>"
            f"Gap-filled Mean: {mean_gap:.2f}; "
            f"Median: {median_gap:.2f}"
        ),
        xref="x domain",
        yref="y domain",
        x=0.01,
        y=0.01,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        align="left",
        font=dict(
            family="Arial",
            size=12,
            color="black",
        ),
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="rgba(80,80,80,0.85)",
        borderwidth=1,
        borderpad=2,
    )

    fig.add_annotation(
        text=(
            f"Spatial Completeness: "
            f"{spatial_completeness:.1f}%"
        ),
        xref="x2 domain",
        yref="y2 domain",
        x=0.01,
        y=0.01,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        align="left",
        font=dict(
            family="Arial",
            size=12,
            color="black",
        ),
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="rgba(80,80,80,0.85)",
        borderwidth=1,
        borderpad=2,
    )

    # ------------------------------
    # Layout
    # ------------------------------

    fig.update_layout(
        template="plotly_white",
        width=600,
        height=400,
        margin=dict(
            l=45,
            r=25,
            t=45,
            b=35,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="#E5ECF6",
        font=dict(
            family="Arial",
            size=9,
            color="black",
        ),
        showlegend=False,
    )

    fig.update_xaxes(
        range=R14_LON_RANGE,
        tickformat=".1f",
        tickfont=dict(size=8),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        gridwidth=1,
        zeroline=False,
        showline=False,
    )

    fig.update_yaxes(
        range=R14_LAT_RANGE,
        tickformat=".1f",
        tickfont=dict(size=8),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        gridwidth=1,
        zeroline=False,
        showline=False,
    )

    fig.update_yaxes(
        showticklabels=False,
        row=1,
        col=2,
    )

    del ntl
    del gap
    del mqf
    del mqf_plot
    del land_mask
    del valid_ntl_mask

    return fig, statistics


# ==========================================================
# Display all four Plotly figures in the notebook
# No dates are added to the figures.
# No files are exported by this block.
# ==========================================================

R14_PREVIEW_ROWS = []

for date_key, metadata in R14_DATES.items():

    preview_fig, preview_statistics = (
        build_r14_figure(date_key)
    )

    preview_fig.show(
        config={
            "displaylogo": False,
            "responsive": True,
        }
    )

    preview_statistics[
        "panel"
    ] = metadata["panel"]

    preview_statistics[
        "label"
    ] = metadata["label"]

    R14_PREVIEW_ROWS.append(
        preview_statistics
    )

    del preview_fig
    gc.collect()

R14_EVIDENCE = (
    pd.DataFrame(R14_PREVIEW_ROWS)
    .sort_values("date")
    .reset_index(drop=True)
)

display(R14_EVIDENCE)

In [ ]:
# Export Figure 3 panels
R14_EXPORT_DIR = OUTPUT_MAIN_FIG_DIR
R14_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for date_key in R14_DATES:
    export_fig, _ = build_r14_figure(date_key)

    export_figure(
        export_fig,
        R14_EXPORT_DIR / f"fig_03_{date_key}",
        width=600,
        height=400,
        scale=3,
    )

    del export_fig
    gc.collect()

print(f"Saved Figure 3 panels to: {R14_EXPORT_DIR.relative_to(PROJECT_ROOT)}")

## Conclusion

The spatial panels and completeness summaries establish the support available for interpreting daily radiance. Continue to Notebook 11 for local supports and Notebook 12 for settlement classes.